In [1]:
import pandas as pd
import numpy as np
import re
import os
from datasets import Dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. 基础配置
model_path = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
metric = evaluate.load("accuracy")

# 2. 数据加载与清洗 (延续之前的特征工程)
df = pd.read_csv('data/train.csv')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_df(input_df):
    temp_df = input_df.copy()
    temp_df['keyword'] = temp_df['keyword'].fillna('unknown')
    temp_df['text'] = temp_df['text'].apply(clean_text)
    # 拼接核心特征
    temp_df['text'] = "keyword: " + temp_df['keyword'].apply(clean_text) + ", text: " + temp_df['text']
    return temp_df.drop(['location', 'keyword'], axis=1, errors='ignore')

train_df_processed = preprocess_df(df)
dataset = Dataset.from_pandas(train_df_processed)

In [2]:
# 3. Tokenization (DeBERTa 建议 128 长度)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 4. 准备训练集和验证集
def transform_dataset(ds):
    ds = ds.remove_columns(["id", "text"])
    ds = ds.rename_column("target", "labels")
    return ds

final_dataset = transform_dataset(tokenized_dataset).train_test_split(test_size=0.1)

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

In [4]:
# 5. 模型初始化与指标计算
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6. 训练参数 (针对 DeBERTa 调优)
# 注意：DeBERTa 对学习率比较敏感，建议设置得比 BERT 稍低
training_args = TrainingArguments(
    output_dir="deberta_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,               # 初始学习率
    per_device_train_batch_size=16,   # RTX 5070 Ti 显存充足
    per_device_eval_batch_size=16,
    num_train_epochs=4,               # 训练 4 轮
    weight_decay=0.05,                # 权重衰减
    warmup_ratio=0.1,                 # 预热步数
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    bf16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["test"],
    compute_metrics=compute_metrics,
)

# 7. 开始训练
trainer.train()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.765812,0.434383
2,0.690780,0.700764,0.565617
3,0.691963,0.689520,0.565617
4,0.685246,0.684649,0.565617


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1716, training_loss=0.6883775582124582, metrics={'train_runtime': 129.776, 'train_samples_per_second': 211.164, 'train_steps_per_second': 13.223, 'total_flos': 1802606167345152.0, 'train_loss': 0.6883775582124582, 'epoch': 4.0})

In [ ]:
# 8. 测试集预测与提交
test_df = pd.read_csv('data/test.csv')
test_ids = test_df['id']
test_df_processed = preprocess_df(test_df)
test_dataset = Dataset.from_pandas(test_df_processed)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)

submission = pd.DataFrame({'id': test_ids, 'target': preds})
os.makedirs('outputs', exist_ok=True)
submission.to_csv('outputs/submission_deberta.csv', index=False)
print("DeBERTa Submission saved successfully!")